# Z3-Python 01b — Du style declaratif LINQ au solveur Z3 en Python

**Navigation** : [<< Z3-Python-01 Introduction](Z3-01-Introduction-Python.ipynb) | [Index](README.md) | [Z3-Python-02 Sudoku >>](Z3-02-Sudoku-Python.ipynb) | [Z3 C# (Z3.Linq) ->](../Z3-Linq2Z3/README.md)

*Compagnon de style déclaratif du module [Z3-Python-01 — Introduction](Z3-01-Introduction-Python.ipynb) : approfondissement optionnel de la lane Python, extension unilatérale documentée (aucun twin C#).*

## Objectifs d'apprentissage

A la fin de ce notebook, vous saurez :
1. **Reconaitre** le style declaratif de specification de contraintes (idiome LINQ `Where` / `from ... where ... select`) utilise par la serie soueur C# Z3.Linq
2. **Traduire** cet idiome declaratif en appels imperatifs a l'API z3-py (`Solver()`, `s.add(...)`, `s.check()`, `s.model()`)
3. **Diagnostiquer** l'insatisfiabilite (`unsat`) et extraire un **noyau d'insatisfiabilite** (unsat core) pour identifier les contraintes en conflit
4. **Modeliser** un problème combinatoire non-trivial (coloration de graphe) ou la puissance declarative du solveur surpasse l'ecriture manuelle d'un backtracking

## Positionnement

Cette serie ([Z3-Python](README.md)) utilise z3-py, le binding Python direct, en style **imperatif-symbolique** : on créé un `Solver`, on ajoute des contraintes, on appelle `check`. La serie soeur [Z3 C# (Z3.Linq)](../Z3-Linq2Z3/README.md) expose une couche **declarative** : les contraintes s'ecrivent comme des predicats LINQ, traduits automatiquement en formules SMT.

Ce notebook fait le pont : il montre que les **deux styles expriment la même intention declarative** (decrire le QUOI, pas le COMMENT), et comment retrouver en Python la lisibilite du DSL C# sans perdre l'acces a l'API complete.
## Source

Ce notebook s'inscrit dans la série ancrée par **Yurichev, Dennis — *SAT/SMT by Example* (édition publique du 12 février 2023, 666 pages)**, Ch. 2 Basics (p. 14) + §3.13.2 Unsat core (p. 90) + §9.1 Map coloring (p. 317) — API du solveur Z3Py (`Solver`, `add`, `check`, `model`), extraction d'un noyau d'insatisfiabilité (`assert_and_track`), et coloration de carte. La carte d'Australie du carnet est la variante classique de la carte d'Europe du livre ; le pont LINQ → pyz3 est propre à la série. Gisement : `G:\Mon Drive\MyIA\IA\Bibliographie IA\Symbolic\Satisfiability and SMT Solvers\2023 - SAT_SMT_by_example.pdf` (16,45 Mo, magic bytes `%PDF`). Pages citées d'après la pagination imprimée du fichier déposé. Voir [Z3-API/README.md — Ouvrage de référence](README.md#ouvrage-de-référence) pour l'index chapitré de la série.

In [1]:
# Imports et verification de l'environnement
from z3 import *

print("Imports OK : z3-solver v" + get_version_string())
print("API disponible : Solver, Optimize, Int, Bool, unsat_core, ...")

Imports OK : z3-solver v4.16.0
API disponible : Solver, Optimize, Int, Bool, unsat_core, ...


## 1. L'idiome declaratif LINQ (C# Z3.Linq) — reference

La serie C# Z3.Linq permet d'ecrire un theoreme comme une requête LINQ. Voici un exemple representatif (cellule de reference, executee dans le notebook C# `01_Linq2Z3_Intro.ipynb`) :

```csharp
using (var ctx = new Z3Context())
{
    var theorem = from t in ctx.NewTheorem<Symbols<int, int, int, int, int>>()
                  where t.X1 - t.X2 >= 1
                  where t.X1 - t.X2 <= 3
                  where t.X1 == (2 * t.X3) + t.X5
                  where t.X3 == t.X5
                  select t;
    var solution = theorem.Solve();
}
```

**Ce qui compte pedagogiquement** : chaque clause `where` est une **contrainte**. L'ordre n'importe pas, la facon dont le solveur resout n'apparait pas. C'est l'essence du **declaratif** : on decrit les relations que doivent satisfaire les inconnues, pas l'algorithme de recherche.

En Python, z3-py n'a pas de sucre syntaxique équivalent au compilateur d'arbres d'expressions LINQ. On exprime donc les mêmes contraintes par des appels explicites a `s.add(...)`. C'est legerement plus verbeux mais tout aussi declaratif sur le fond — et on garde l'acces a l'API complete (tactiques, `Optimize`, théories avancees).

## 2. Traduction en pyz3 : un système SAT

Reproduisons en Python le theoreme ci-dessus (5 entiers `X1..X5`, 4 contraintes). Le patron d'usage z3-py :

1. **Créer** un `Solver()`
2. **Declarer** les variables typees (`Int`)
3. **Ajouter** chaque contrainte avec `s.add(...)`
4. **Verifier** avec `s.check()`, puis `s.model()` pour extraire un modèle

In [2]:
# Exemple 1 : theoreme a 5 entiers, 4 contraintes (port direct du theoreme LINQ ci-dessus)
s = Solver()
X1, X2, X3, X4, X5 = Ints('X1 X2 X3 X4 X5')

# Chaque clause 'where' du C# devient un s.add(...) en Python
s.add(X1 - X2 >= 1)            # where t.X1 - t.X2 >= 1
s.add(X1 - X2 <= 3)            # where t.X1 - t.X2 <= 3
s.add(X1 == 2 * X3 + X5)       # where t.X1 == (2 * t.X3) + t.X5
s.add(X3 == X5)                # where t.X3 == t.X5

resultat = s.check()
print("Resultat du solveur :", resultat)
if resultat == sat:
    m = s.model()
    print("Modele trouve :")
    for v in [X1, X2, X3, X4, X5]:
        val = m[v]
        print("  %s = %s" % (v, val if val is not None else "(libre, non assignee par le solveur)"))
    # X4 est libre (aucune contrainte) : le solveur ne l'assigne pas dans le modele
    print("\nNote : X4 n'apparait dans aucune contrainte -> le modele le laisse non assigne (None).")

Resultat du solveur : sat
Modele trouve :
  X1 = 3
  X2 = 2
  X3 = 1
  X4 = (libre, non assignee par le solveur)
  X5 = 1

Note : X4 n'apparait dans aucune contrainte -> le modele le laisse non assigne (None).


### Interpretation : correspondance LINQ <-> pyz3

| C# Z3.Linq (declaratif) | Python z3-py (imperatif-symbolique) |
|-------------------------|--------------------------------------|
| `ctx.NewTheorem<Symbols<...>>()` | `X1, X2, ... = Ints('X1 X2 ...')` |
| `where t.X1 - t.X2 >= 1` | `s.add(X1 - X2 >= 1)` |
| `theorem.Solve()` | `s.check()` + `s.model()` |
| `solution.X1` | `m[X1]` |

Le **contenu logique est identique** : les deux formulations declarent les mêmes 4 contraintes. La différence est purement syntaxique — le compilateur C# traduit l'arbre d'expression LINQ, tandis qu'en Python on ecrit les appels `s.add` explicitement. En contrepartie, z3-py nous donne acces aux tactiques, a `Optimize`, et aux théories de bas niveau que la couche LINQ n'expose pas.

## 3. Diagnostiquer l'insatisfiabilite : unsat et noyau

Cas d'école : une variable doit etre a la fois `> 5` et `< 3` — c'est impossible. Le solveur repond `unsat`. Le **noyau d'insatisfiabilite** (`unsat_core`) identifie quelles contraintes, parmi un ensemble etiquete, sont responsables du conflit : c'est un outil de diagnostic indispensable sur de gros modèles.

Pour obtenir un noyau, on etiquete chaque contrainte avec un `Bool` et on utilise `assert_and_track` au lieu de `add`.

In [3]:
# Exemple 2 : systeme UNSAT + extraction du noyau d'insatisfiabilite
s2 = Solver()
x = Int('x')
p1, p2 = Bools('p1 p2')   # etiquettes logiques des deux contraintes

# assert_and_track(constraint, label) : la contrainte est ajoutee sous une etiquette
s2.assert_and_track(x > 5, p1)   # 'where t.X1 > 5'
s2.assert_and_track(x < 3, p2)   # 'where t.X1 < 3'

resultat = s2.check()
print("Contraintes : x > 5  ET  x < 3")
print("Resultat du solveur :", resultat)
if resultat == unsat:
    noyau = s2.unsat_core()
    print("Noyau d'insatisfiabilite :", noyau)
    print("-> Les contraintes etiquetees", [str(b) for b in noyau],
          "sont conjointement incompatibles.")

Contraintes : x > 5  ET  x < 3
Resultat du solveur : unsat
Noyau d'insatisfiabilite : [p2, p1]
-> Les contraintes etiquetees ['p2', 'p1'] sont conjointement incompatibles.


### Interpretation : sat vs unsat vs unknown

| Reponse | Sens | Utilite du noyau |
|---------|------|------------------|
| `sat` | Au moins une assignation satisfait tout | `s.model()` donne un exemple |
| `unsat` | Aucune assignation possible | `unsat_core()` localise les contraintes fautives |
| `unknown` | Le solveur n'a pas tranche (trop dur / timeout) | Pas de noyau ; affiner le modèle ou augmenter le budget |

Le noyau d'insatisfiabilite est l'équivalent logique d'un **message d'erreur précis** : au lieu de dire « votre modèle est impossible », il pointe les contraintes (parfois 2 ou 3 parmi des centaines) qui se contredisent. Sur un problème industriel (planning, verification), c'est ce qui rend le debuggage envisageable.

## 4. Un problème combinatoire non-trivial : coloration de graphe

Pour montrer ou le paradigme declaratif prend tout son sens, modelisons un **vrai** problème combinatoire : la **coloration de carte** (combien de couleurs suffisent pour que deux regions frontieres aient des couleurs différentes ?). C'est le coeur du theoreme des 4 couleurs.

Nous colorions la carte de l'Australie (6 etats continentaux contigus) avec **3 couleurs**. Construire un backtracking manuel pour ce problème est fastidieux (gestion des conflits, ordre d'affectation, retour-arriere). En z3-py, on se contente de **declarer l'adjacence** et le solveur trouve une coloration valide.

**Adjacences** (frontieres terrestres) :
- Australie-Occidentale (WA) : NT, SA
- Territoire du Nord (NT) : WA, SA, Q
- Australie-Meridionale (SA) : WA, NT, Q, NSW, V
- Queensland (Q) : NT, SA, NSW
- Nouvelle-Galles du Sud (NSW) : SA, Q, V
- Victoria (V) : SA, NSW

(Tasmanie est insulaire, hors du graphe continental.)

In [4]:
# Exemple 3 : coloration de la carte d'Australie avec 3 couleurs
couleurs = ['rouge', 'vert', 'bleu']                # 3 couleurs
etats = ['WA', 'NT', 'SA', 'Q', 'NSW', 'V']

# Une variable Int par etat, domanee a {0,1,2} (indices de couleurs)
c = {e: Int(e) for e in etats}

s3 = Solver()
for e in etats:
    s3.add(c[e] >= 0, c[e] <= 2)                    # domaine : 3 couleurs

# Adjacences : deux etats frontaliers ont des couleurs differentes
adjacences = [('WA','NT'), ('WA','SA'), ('NT','SA'), ('NT','Q'),
              ('SA','Q'), ('SA','NSW'), ('SA','V'), ('Q','NSW'), ('NSW','V')]
for a, b in adjacences:
    s3.add(c[a] != c[b])                            # contrainte 'deux voisins differents'

resultat = s3.check()
print("Coloration 3-couleurs de l'Australie :", resultat)
if resultat == sat:
    m = s3.model()
    print("\nColoration valide :")
    for e in etats:
        idx = m[c[e]].as_long()
        print("  %-4s -> %s" % (e, couleurs[idx]))
    # Verification automatique : aucune paire adjacente n'a la meme couleur
    conflits = [(a, b) for a, b in adjacences if m[c[a]].as_long() == m[c[b]].as_long()]
    print("\nVerification : %d conflit(s) adjacent(s) -> %s" % (len(conflits), 'OK' if not conflits else 'BUG'))

Coloration 3-couleurs de l'Australie : sat

Coloration valide :
  WA   -> bleu
  NT   -> vert
  SA   -> rouge
  Q    -> bleu
  NSW  -> vert
  V    -> bleu

Verification : 0 conflit(s) adjacent(s) -> OK


### Interpretation : pourquoi le declaratif gagne ici

Essayez d'imaginer le code **imperatif** équivalent : il faudrait choisir un ordre d'affectation des 6 etats, implementer un backtracking avec detection de conflit, gerer l'expiration des branches mortes, et esperer que l'heuristique d'ordre soit bonne. Pour 6 etats c'est faisable ; pour 60 (carte d'un grand pays avec toutes ses subdivisions), c'est un enfer de maintenance.

Avec z3-py, le **même patron** (declarer le domaine + declarer l'adjacence + `check`) resout n'importe quelle taille d'instance. Ajouter un etat = ajouter une variable et quelques contraintes d'adjacence ; l'algorithme de resolution, lui, ne change jamais. C'est le gain fondamental du paradigme declaratif : la **complexite algorithmique est deleguee au solveur**, le modèle exprime seulement la structure du problème.

> **Lecon** : le seuil ou le declaratif surpasse l'imperatif n'est pas le nombre de variables, mais la **richesse des contraintes**. Des qu'apparait une combinaison de contraintes (adjacence + cardinalite + optimisation), ecrire un backtracking correct devient plus difficile que de declarer les contraintes.

## 5. Exemples guidés — trois résolutions complètes

Les trois énoncés de cette section sont résolus de bout en bout : déclarer les inconnues, poser chaque contrainte, interroger le solveur, lire le verdict. Le code de chaque résolution est conservé dans son état d'origine ; le bloc « Lecture du résultat » qui suit chaque cellule explique ce que la sortie prouve. Les exercices à faire vous-même attendent en section 6.

### Exemple guidé 1 — Traduire un théorème LINQ en pyz3 (crédité #25495 — Emmajvx)

> Résolution originale : **Emmajvx** (groupe 2), rendu de TP
> [#25495](https://github.com/jsboige/CoursIA/pull/25495). Reprise telle quelle ;
> seule la mise en forme de l'énoncé est harmonisée avec les sections voisines.

Soit le théorème C# (4 entiers) :
```csharp
where t.X1 + t.X2 == t.X3
where t.X3 == t.X4 * 2
where t.X1 >= 1
```

Les inconnues se déclarent par `Ints(...)`, chaque clause `where` devient un `s.add(...)`, puis `check()` et `model()` livrent un témoin.

In [5]:
s_ex1 = Solver()
X1, X2, X3, X4 = Ints('X1 X2 X3 X4')
s_ex1.add(X1 + X2 == X3)
s_ex1.add(X3 == X4 * 2)
s_ex1.add(X1 >= 1)

resultat = s_ex1.check()
print("Résultat :", resultat)
if resultat == sat:
    m = s_ex1.model()
    for v in [X1, X2, X3, X4]:
        print(v, "=", m[v])

Résultat : sat
X1 = 1
X2 = 1
X3 = 2
X4 = 1


### Lecture du résultat : un modèle, pas *le* modèle

Le solveur répond `sat` et propose `X1 = 1, X2 = 1, X3 = 2, X4 = 1`. Le système n'a pas de solution unique : la contrainte `X3 == X4 * 2` fixe `X4` dès que `X3` est choisi, mais le couple `(X1, X2)` peut encore varier. Comparez avec l'Exemple 1 de la section 2, où une variable entièrement libre restait non assignée : chaque modèle est un **témoin** parmi d'autres, jamais « la » réponse. C'est le sens du déclaratif — l'énoncé décrit l'espace des solutions, le solveur en exhibe un point.

### Exemple guidé 2 — Détecter un UNSAT et extraire le noyau (crédité #25495 — Emmajvx)

> Résolution originale : **Emmajvx** (groupe 2), rendu de TP
> [#25495](https://github.com/jsboige/CoursIA/pull/25495). Reprise telle quelle ;
> seule la mise en forme de l'énoncé est harmonisée avec les sections voisines.

Trois contraintes sur un entier `n` :

1. `n` est pair (`n % 2 == 0`)
2. `n > 10`
3. `n < 8`

Chacune est posée avec `assert_and_track` sous une étiquette `Bool` (`e1`, `e2`, `e3`), puis le `unsat_core` nomme celles qui se contredisent. La parité s'écrit directement `n % 2 == 0` — l'arithmétique entière de Z3 accepte le modulo.

In [6]:
s_ex2 = Solver()
n = Int('n')
e1, e2, e3 = Bools('e1 e2 e3')

s_ex2.assert_and_track(n % 2 == 0, e1)   # n est pair
s_ex2.assert_and_track(n > 10, e2)       # n > 10
s_ex2.assert_and_track(n < 8, e3)        # n < 8

resultat = s_ex2.check()
print("Résultat :", resultat)
if resultat == unsat:
    print("Noyau :", s_ex2.unsat_core())

Résultat : unsat
Noyau : [e2, e3]


### Lecture du résultat : le noyau acquitte une contrainte

Le verdict est `unsat` et le noyau rendu contient exactement `e2` et `e3` (l'ordre d'affichage varie d'une exécution à l'autre) : les contraintes `n > 10` et `n < 8` se contredisent à elles deux — aucun entier n'est à la fois plus grand que 10 et plus petit que 8. La contrainte de parité `e1` est **acquittée** : le solveur n'en a pas besoin pour prouver le conflit. Retirez mentalement `e2` : `n = 6` satisfait le reste ; retirez `e3` : `n = 12` suffit. C'est tout l'intérêt du noyau sur un vrai modèle : il accuse des contraintes précises au lieu de dire « impossible » en bloc.

### Exemple guidé 3 — La carte d'Australie est-elle 2-coloriable ? (crédité #25495 — Emmajvx, #25618 — MathisGramaglia)

> Deux résolutions indépendantes et équivalentes, rendus de TP : **Emmajvx** (groupe 2,
> [#25495](https://github.com/jsboige/CoursIA/pull/25495)) et **MathisGramaglia** (groupe 4,
> [#25618](https://github.com/jsboige/CoursIA/pull/25618)). Le code ci-dessous reprend celle de #25495
> telle quelle ; seule la mise en forme de l'énoncé est harmonisée avec les sections voisines.

La question de décision reprend le modèle de coloration de la section 4, avec le domaine réduit à `{0, 1}` : existe-t-il une coloration valide avec seulement 2 couleurs ? Le verdict du `check` — `sat` ou `unsat` — est la réponse, et un `unsat` prouve que 2 couleurs ne suffisent pas.

In [7]:
from z3 import *

s_ex3 = Solver()
etats = ['WA', 'NT', 'SA', 'Q', 'NSW', 'V']
c = {e: Int(e) for e in etats}

for e in etats:
    s_ex3.add(c[e] >= 0, c[e] <= 1)   # 2 couleurs

adjacences = [('WA','NT'), ('WA','SA'), ('NT','SA'), ('NT','Q'),
              ('SA','Q'), ('SA','NSW'), ('SA','V'), ('Q','NSW'), ('NSW','V')]
for a, b in adjacences:
    s_ex3.add(c[a] != c[b])

resultat = s_ex3.check()
print("2-coloriable ?", resultat)
if resultat == sat:
    print(s_ex3.model())
else:
    print("Non : 2 couleurs ne suffisent pas, il en faut au moins 3.")

2-coloriable ? unsat
Non : 2 couleurs ne suffisent pas, il en faut au moins 3.


### Lecture du résultat : `unsat` comme preuve d'impossibilité

Avec deux couleurs, le verdict `unsat` n'est pas un échec du solveur : c'est une **preuve** qu'aucune coloration n'existe. La raison tient en trois États : `WA`, `NT` et `SA` sont deux à deux frontaliers — un triangle — or un graphe 2-coloriable (biparti) ne contient jamais de triangle. Combiné à l'Exemple 3 de la section 4 (`sat` avec 3 couleurs), le verdict fixe le nombre exact de couleurs nécessaires pour l'Australie : trois. C'est le premier pas vers la question « combien de couleurs suffisent pour un graphe donné ? » — l'Exercice 4 la pose pour un autre graphe.

## 6. Exercices

Trois exercices, cette fois non résolus. Chaque stub est incomplet (marqué `# TODO etudiant`) : à vous d'écrire les contraintes. Les cellules s'exécutent sans erreur même non complétées (elles affichent un message d'attente).

### Exercice 4 — Le nombre chromatique d'une roue, par interrogation répétée du solveur

L'Exemple guidé 3 a répondu à « l'Australie est-elle 2-coloriable ? » par un unique `check`. La question qui généralise : **combien de couleurs suffisent, au minimum, pour un graphe donné ?** — son **nombre chromatique**.

Prenez la roue à cinq rayons : un sommet central relié aux cinq sommets d'un cycle (le centre voisine avec `v0` à `v4`, et le cycle ferme la boucle `v0–v1–v2–v3–v4–v0`). Le nombre de couleurs n'est plus une donnée mais l'inconnue : pour chaque `k` croissant à partir de 1, bornez chaque sommet au domaine `{0, ..., k-1}`, ajoutez les contraintes d'adjacence, et interrogez le solveur.

**Indice** : le premier `k` pour lequel le solveur répond `sat` est le nombre chromatique ; les `unsat` précédents prouvent que les `k` plus petits ne suffisent pas. La cellule d'imports de ce carnet cite aussi `Optimize` — un `Optimize()` où `k` est une variable à `minimize` pose la même question en un seul appel.

**Étape 1** : construisez la liste des dix arêtes (cinq du cycle, cinq des rayons), puis bouclez sur `k`.

In [8]:
# Exercice 4 : le nombre chromatique de la roue a cinq rayons (etudiant)
s_ex4 = Solver()
# sommets = ...   # TODO etudiant : le centre et v0..v4
# aretes  = ...   # TODO etudiant : 5 aretes du cycle + 5 rayons vers le centre
# TODO etudiant : pour k = 1, 2, 3, ... bornez chaque sommet a {0..k-1} + contraintes d'adjacence, puis s_ex4.check()
# TODO etudiant : le premier k satisfiable est le nombre chromatique -- affichez-le

print("Exercice 4 a completer : retirez les commentaires TODO, bouclez sur k croissant et affichez le nombre chromatique.")

Exercice 4 a completer : retirez les commentaires TODO, bouclez sur k croissant et affichez le nombre chromatique.


### Exercice 5 — Compter toutes les colorations valides par blocage de modèle

Un `check()` satisfait ne rend qu'**un** modèle : un témoin parmi d'autres. Pour les énumérer tous, on **bloque** chaque modèle trouvé — on ajoute la contrainte qui l'exclut, `Or` des différences sur toutes les variables — et on recommence jusqu'à `unsat`.

Reprenez le modèle de coloration 3-couleurs de l'Australie (section 4) et comptez le nombre total de colorations valides.

**Indice** : dans la boucle, `m[c[e]].as_long()` rend la couleur du modèle pour chaque État, et la contrainte de blocage s'écrit `Or([c[e] != valeur pour e in etats])`. Comparez le total au nombre de permutations des trois couleurs : que dit l'écart sur la structure du graphe ?

**Étape 1** : affichez chaque coloration trouvée avant de la bloquer — vous verrez le solveur épuiser les possibilités une à une.

In [9]:
# Exercice 5 : compter les 3-colorations de l'Australie par blocage de modeles (etudiant)
s_ex5 = Solver()
# c = {e: Int(e) for e in ['WA','NT','SA','Q','NSW','V']}   # TODO etudiant : reprendre le modele de la section 4
# TODO etudiant : domaine {0,1,2} + les 9 contraintes d'adjacence
# compteur = 0
# TODO etudiant : tant que s_ex5.check() == sat : lire le modele, compteur += 1,
#                 puis bloquer le modele trouve (Or des differences sur les 6 etats)

print("Exercice 5 a completer : retirez les commentaires TODO et enumerez les colorations en bloquant chaque modele trouve.")

Exercice 5 a completer : retirez les commentaires TODO et enumerez les colorations en bloquant chaque modele trouve.


### Exercice 6 — Diagnostiquer un emploi du temps impossible

Trois cours `C1`, `C2`, `C3` occupent chacun un créneau parmi `1`, `2`, `3`. Cinq contraintes :

1. `C1` et `C2` ne partagent pas de créneau (même enseignant)
2. `C1` est au créneau `1` (disponibilité de l'intervenant)
3. `C2` est au créneau `1` (la salle n'est libre qu'au premier créneau)
4. `C3` est au créneau `2` ou `3` (le créneau `1` est pris par un amphi transversal)
5. si `C3` est au créneau `2`, alors `C1` est au créneau `3` (l'amphi du créneau 2 mobilise aussi le groupe de `C1`)

L'ensemble est insatisfiable. Posez les cinq contraintes avec `assert_and_track` et des étiquettes `e1` à `e5`, puis extrayez le noyau.

**Indice** : la contrainte 5 est une implication — `Implies(t['C3'] == 2, t['C1'] == 3)` se pose telle quelle dans Z3. Le noyau rendu est un **sous-ensemble** des étiquettes qui suffit, à lui seul, à rendre le problème insatisfiable : les contraintes absentes du noyau ne sont pas nécessaires au conflit. Pour savoir si ce noyau est minimal, retirez-en les contraintes une à une : si chaque retrait rend le problème satisfiable, aucune n'est superflue.

**Étape 1** : déclarez les créneaux `t = {x: Int(x) for x in ['C1','C2','C3']}` bornés à `{1, 2, 3}`, puis étiquetez les cinq contraintes.

In [10]:
# Exercice 6 : noyau d'insatisfiabilite d'un emploi du temps (etudiant)
s_ex6 = Solver()
# t = {x: Int(x) for x in ['C1','C2','C3']}      # TODO etudiant : creneau de chaque cours, domaine {1,2,3}
# e1, e2, e3, e4, e5 = Bools('e1 e2 e3 e4 e5')   # TODO etudiant : etiquettes
# TODO etudiant : poser les 5 contraintes avec s_ex6.assert_and_track (la 5e est une Implies)
# TODO etudiant : s_ex6.check() puis s_ex6.unsat_core() -- quelles etiquettes le solveur acquitte-t-il ?

print("Exercice 6 a completer : retirez les commentaires TODO, etiquetez les 5 contraintes et lisez le noyau.")

Exercice 6 a completer : retirez les commentaires TODO, etiquetez les 5 contraintes et lisez le noyau.


## 7. Synthese

| Concept | Cote C# (Z3.Linq) | Cote Python (z3-py) |
|---------|-------------------|---------------------|
| **Style** | Declaratif (LINQ `Where`/`from`) | Imperatif-symbolique (`Solver.add`) |
| **Specification** | Arbres d'expressions compiles | Appels `s.add` explicites |
| **Avantage** | Lisibilite proche du pseudo-code | Acces complet (tactiques, Optimize, théories) |
| **Insatisfiabilite** | `theorem.Solve()` leve / retourne | `check() == unsat` + `unsat_core()` |

**Ce qu'il faut retenir** :
- Les deux styles sont **equivalents en expressivite** sur les problemes de satisfaction ; le choix est une question de lisibilite vs contrôle.
- Le **noyau d'insatisfiabilite** (`unsat_core`) est l'outil de debuggage essentiel : il localise les contraintes fautives dans un modèle.
- Le paradigme declaratif prend tout son sens sur les **problemes combinatoires** (coloration, planning, allocation) : on declare la structure, le solveur resout. La taille de l'instance n'affecte que les performances, jamais la structure du code.

**Pour aller plus loin** :
- [Z3-Python-06 — Optimisation avancee](Z3-06-Advanced-Optimization-Python.ipynb) : `Optimize`, `maximize`/`minimize`, quand la satisfaction devient optimisation sous objectifs.
- [Z3 C# (Z3.Linq)](../Z3-Linq2Z3/README.md) : la serie soeur declarative, pour comparer les idiomes dans leur contexte natif.
- La documentation [z3-solver (pyz3)](https://z3prover.github.io/api/html/namespacez3py.html) pour l'API complete.